In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys
import datetime
from tqdm import tqdm

%load_ext autoreload
%autoreload 2

sys.path.append('/home/wolfgang/repos/sleep_general')
from mgh_sleeplab import load_mgh_signal, annotations_preprocess, vectorize_respiratory_events, vectorize_sleep_stages

In [2]:
table_grass = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/grass_studies_list.csv')
table_natus = pd.read_csv('/media/mad3/Datasets_ConvertedData/sleeplab/natus_studies_list.csv')
assert np.all(table_grass.columns == table_natus.columns)
sleeplab_table = pd.concat([table_grass, table_natus], axis=0)
sleeplab_table = sleeplab_table[pd.notna(sleeplab_table.MRN)]
sleeplab_table.reset_index(inplace=True, drop=True)
print(sleeplab_table.shape)

(25231, 10)


In [3]:
print('TMP')
part = 1
sleeplab_table = sleeplab_table.iloc[8700*part : 8700*(part+1)].copy()
print(sleeplab_table.shape)

TMP
(8700, 10)


In [4]:
sleeplab_table['path_signal'] = np.nan
sleeplab_table['path_annotations'] = np.nan

for jloc, row in sleeplab_table.iterrows():
    
    try:
        path = row.Path
        path = path.replace('M:', '/media/mad3')
        path = path.replace('\\', '/')

        path_signalfile = np.nan
        signalfile = [x for x in os.listdir(path) if 'Signal' in x]
        if len(signalfile) > 0:
            signalfile = signalfile[0]
            path_signalfile = os.path.join(path, signalfile)

        path_annotations = np.nan
        if 'annotations.csv' in os.listdir(path):
            path_annotations = os.path.join(path, 'annotations.csv')

        sleeplab_table.loc[jloc, 'path_signal'] = path_signalfile
        sleeplab_table.loc[jloc, 'path_annotations'] = path_annotations

    except Exception as e:
        print(e)
        continue
        
sleeplab_table.to_csv(f'sleeplab_table_{part}.csv')

In [5]:
print('TMP')
sleeplab_table = sleeplab_table.iloc[:8000].copy()
print(sleeplab_table.shape)

TMP
(8000, 12)


In [6]:
from sleep_analysis_functions import compute_spo2_clean, compute_hypoxia_burden, desaturation_detection, hypoxaemic_burden_minutes

def compute_hypoxia_statistics(data, apnea, sleep_stage, fs):
    
    data['apnea'] = apnea

    dt_start = pd.Timestamp('2000-01-01 00:00:00')
    dt_end = dt_start + datetime.timedelta(seconds=(data.shape[0]-1) / fs)
    pseudo_dt_index = pd.date_range(start=dt_start, end=dt_end, periods=data.shape[0])
    data.index = pseudo_dt_index

    data = compute_spo2_clean(data, fs=fs)
    data['spo2'] = data['spo2_clean']

    data['apnea_binary'] = np.isin(data['apnea'],[1,2,3,4]).astype(int)
    data['apnea_end'] = np.isin(data['apnea_binary'].diff(), [-1])
    
    sleep_stage = sleep_stage[np.logical_not(pd.isna(sleep_stage))]
    hours_sleep = sum(sleep_stage<5)/fs/3600
    
    data, hypoxia_burden = compute_hypoxia_burden(data, fs, hours_sleep=hours_sleep, apnea_name = 'apnea')
    
    T90burden, T90desaturation, T90nonspecific = hypoxaemic_burden_minutes(data['spo2'].values, fs)
    
    AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)

    return hypoxia_burden, AHI, hours_sleep, T90burden, T90desaturation, T90nonspecific


In [7]:
sleeplab_table.shape

(8000, 12)

In [12]:
sleeplab_table['Total_Sleep_Time'] = np.nan
sleeplab_table['AHI'] = np.nan
sleeplab_table['HypoxiaBurden'] = np.nan
sleeplab_table['T90_minutes'] = np.nan
sleeplab_table['T90_minutes_desat'] = np.nan
sleeplab_table['T90_minutes_unspecific'] = np.nan

for jloc, row in sleeplab_table.iterrows():
    
    try:
        if pd.isna(row.path_signal) | pd.isna(row.path_annotations):
            continue
        # read in raw data:
        signal, params = load_mgh_signal(row.path_signal)
        annotations = pd.read_csv(row.path_annotations)

        fs = int(params['Fs'])
        signal_len = signal.shape[0]

        # get respiratory event and sleep stage array:
        annotations = annotations_preprocess(annotations, fs)
        resp = vectorize_respiratory_events(annotations, signal_len)
        sleep_stage = vectorize_sleep_stages(annotations, signal_len)

        hypoxia_burden, ahi, total_sleep_time, T90burden, T90desaturation, T90nonspecific = compute_hypoxia_statistics(signal, resp, sleep_stage, fs)

        sleeplab_table.loc[jloc, 'AHI'] = ahi
        sleeplab_table.loc[jloc, 'HypoxiaBurden'] = hypoxia_burden
        sleeplab_table.loc[jloc, 'Total_Sleep_Time'] = total_sleep_time
        sleeplab_table.loc[jloc, 'T90_minutes'] = T90burden
        sleeplab_table.loc[jloc, 'T90_minutes_desat'] = T90desaturation
        sleeplab_table.loc[jloc, 'T90_minutes_unspecific'] = T90nonspecific

        sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')
        
    except Exception as e:
        print(f'{jloc}, {row.Path}')
        print(e)

sleeplab_table.to_csv(f'sleeplab_table_hypoxia_{part}.csv')

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:479: RuntimeWarning: invalid value encountered in double_scalars
  hypoxic_burden = np.sum(areas_robust)/hours_sleep


8712, M:\Datasets_ConvertedData\sleeplab\grass_data\Oil_Karen_121710_2245.000
float division by zero


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

8862, M:\Datasets_ConvertedData\sleeplab\grass_data\Mahoney_Truman_012014_2336.000
Cannot convert non-finite values (NA or inf) to integer


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,


...

/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
<ipython-input-6-219ea860a567>:25: RuntimeWarning: invalid value encountered in double_scalars
  AHI = np.round(sum(data['apnea_end'])/hours_sleep,1)
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/pyt

16493, M:\Datasets_ConvertedData\sleeplab\grass_data\Hansen_Tor_080714_0855.000
index 4354400 is out of bounds for axis 0 with size 4354400


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

16563, M:\Datasets_ConvertedData\sleeplab\grass_data\McCooey_Candice_031711_0843.000
index 5869000 is out of bounds for axis 0 with size 5869000
16569, M:\Datasets_ConvertedData\sleeplab\grass_data\Copy of Marulli_Ralph_022010_2141.000
float division by zero


/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,
/home/wolfgang/anaconda3/envs/analysis/lib/python3.8/site-packages/numpy/lib/nanfunctions.py:1389: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a

In [9]:
sleeplab_table

,FolderName,PatientID,MRN,LastName,FirstName,Sex,DateOfBirth,DateOfVisit,TypeOfTest,Path,path_signal,path_annotations,Total_Sleep_Time,AHI,HypoxiaBurden,T90_minutes,T90_minutes_desat,T90_minutes_unspecific
8700,Martin_James_100216_2148.000,Z10528677,439-75-73,Martin,James D,Male,1968-11-16,2016-10-02,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,8.141667,5.4,8.1,1.032417,1.032417,0.000000
8701,Bravo_Elizabeth_021116_2156.000,Z8848742,448-75-41,Bravo,Elizabeth,Female,1990-10-10,2016-02-11,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,7.775000,0.4,0.0,0.172167,0.000000,0.172167
8702,NIERENBERG_MINDY_051210_2216.001,Z6841052,219-49-18,Nierenberg,Mindy R.,Female,1955-12-14,2010-05-12,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8703,Wharff_shirley_072010_2228.000,Z8578934,116-65-45,Wharff,Shirley D,Female,1928-02-21,2010-07-20,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,6.941944,11.5,21.0,3.831500,3.398583,0.432917
8704,Lascuola_TheresaC_080208_2214.000,Z9355256,391-07-43,Lascuola,Theresa C,Female,1959-01-28,2008-08-02,PSG all night CPAP,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16695,Lee_David_072210_2254.000,Z9718323,416-33-39,Lee,David A,Male,1946-01-08,2010-07-22,PSG Split night,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN
16696,Shapiro_Edward K_120116_2242.000,Z11767424,345-25-61,Shapiro,Edward K.,Male,1937-03-03,2016-12-01,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN
16697,Martin_James_110313_2058.000,Z10528677,439-75-73,Martin,James D,Male,1968-11-16,2013-11-03,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN
16698,Abduljabar_Rafah_041614_2338.000,Z8419447,500-11-22,Abduljabar,Rafah,Female,1978-05-08,2014-04-16,PSG Diagnostic,M:\Datasets_ConvertedData\sleeplab\grass_data\...,/media/mad3/Datasets_ConvertedData/sleeplab/gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# fig, ax = plt.subplots(4, 1, figsize=(10, 8), sharex=True)

# ax[0].plot(resp)
# ax[0].plot(sleep_stage)
# ax[1].plot(signal['spo2'])
# ax[1].set_ylim([80,100])
# # ax[2].plot(signal['ptaf'])
# ax[3].plot(signal['chest'])
# ax[3].plot(signal['abd'])